# 📧 Outlook Email Generator from Excel Report
Reads your Excel tracker, filters **In Progress** tasks, and builds a ready-to-paste email body + subject.

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Update these before running

EXCEL_FILE      = "report.xlsx"        # Path to your Excel file
SHEET_NAME      = 0                    # Sheet index (0 = first) or sheet name string

# Column names exactly as they appear in your Excel header row
COL_TASK        = "Tasks"
COL_STATUS      = "Status"
COL_UPDATE      = "Update from primary owner"

# Status value to filter (case-insensitive)
FILTER_STATUS   = "In Progress"

# ── EMAIL PLACEHOLDERS ────────────────────────────────────────────────────────
# These are pulled from the system / set manually — edit as needed each run
import datetime, getpass, socket

SENDER_NAME     = getpass.getuser().replace(".", " ").title()   # auto: OS login name
TODAY           = datetime.date.today().strftime("%d %b %Y")    # auto: today's date
TEAM_NAME       = "Operations Team"                             # ← change as needed
REPORT_PERIOD   = "Weekly"                                      # Daily / Weekly / etc.

# ── ADDITIONAL BODY SECTIONS ──────────────────────────────────────────────────
# Add your custom intro / footer text here (HTML supported)
EMAIL_INTRO = """
Hi Team,<br><br>
Please find below the <b>{period} Status Update</b> as of <b>{date}</b>.
The following tasks are currently <b>In Progress</b>:
"""

EMAIL_FOOTER = """
<br>
Please reach out if you have any questions or blockers.<br><br>
Thanks & Regards,<br>
<b>{sender}</b><br>
{team}
"""
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import pandas as pd
from IPython.display import display, HTML

# ── READ EXCEL ────────────────────────────────────────────────────────────────
df = pd.read_excel(EXCEL_FILE, sheet_name=SHEET_NAME)
df.columns = df.columns.str.strip()          # remove accidental whitespace

print(f"✅ Loaded {len(df)} rows from '{EXCEL_FILE}'")
print(f"   Columns found: {list(df.columns)}")
df.head()

In [ ]:
# ── FILTER IN-PROGRESS TASKS ─────────────────────────────────────────────────
mask = df[COL_STATUS].astype(str).str.strip().str.lower() == FILTER_STATUS.lower()
df_filtered = df[mask][[COL_TASK, COL_UPDATE]].reset_index(drop=True)

print(f"📋 {len(df_filtered)} task(s) with status '{FILTER_STATUS}'")
df_filtered

In [ ]:
# ── BUILD EMAIL ───────────────────────────────────────────────────────────────

def build_email(df_in, intro_tmpl, footer_tmpl):
    """Returns (subject, plain_text, html_body)"""

    subject = f"{REPORT_PERIOD} Status Update – {TODAY} | {TEAM_NAME}"

    intro   = intro_tmpl.format(period=REPORT_PERIOD, date=TODAY)
    footer  = footer_tmpl.format(sender=SENDER_NAME, team=TEAM_NAME)

    # ── HTML body
    html_rows = ""
    plain_rows = ""

    for _, row in df_in.iterrows():
        task   = str(row[COL_TASK]).strip()
        update = str(row[COL_UPDATE]).strip()
        update = update if update.lower() not in ("", "nan", "none") else "No update provided."

        html_rows  += f"<li><b>{task}</b> : {update}</li>\n"
        plain_rows += f"  • {task} : {update}\n"

    html_body = f"""
<html><body style="font-family:Calibri,Arial,sans-serif; font-size:14px; color:#202020;">
{intro}
<ul style="line-height:1.8;">
{html_rows}
</ul>
{footer}
</body></html>"""

    plain_body = (
        intro.replace("<br>","\n").replace("<b>","").replace("</b>","") + "\n\n"
        + plain_rows + "\n"
        + footer.replace("<br>","\n").replace("<b>","").replace("</b>","")
    )

    return subject, plain_body, html_body


subject, plain_body, html_body = build_email(df_filtered, EMAIL_INTRO, EMAIL_FOOTER)

print("=" * 60)
print("SUBJECT:")
print(subject)
print("=" * 60)

In [ ]:
# ── PLAIN-TEXT PREVIEW (copy-paste friendly) ──────────────────────────────────
print("PLAIN TEXT BODY (copy & paste into Outlook):\n")
print(plain_body)

In [ ]:
# ── HTML PREVIEW (rendered) ───────────────────────────────────────────────────
print("HTML PREVIEW:")
display(HTML(html_body))

In [ ]:
# ── SAVE HTML TO FILE (open in browser → Copy All → Paste into Outlook) ───────
out_file = f"email_draft_{datetime.date.today().strftime('%Y%m%d')}.html"
with open(out_file, "w", encoding="utf-8") as f:
    f.write(html_body)

print(f"💾 HTML draft saved to: {out_file}")
print("   Tip: Open the HTML file in Chrome/Edge → Ctrl+A → Ctrl+C → Paste into Outlook message body")